In [2]:
import numpy as np

In [ ]:
class HestonModelSimulator:
    def __init__(self, params: HestonParams, config: MonteCarloConfig):
        # Validate configurations before starting simulation
        params.validate()
        config.validate()

        self.params = params
        self.config = config

In [ ]:
 def simulate(self):
        """
        Simulates asset and variance paths using a Full Truncation Euler Scheme.
        """
        # Extract parameters for readability
        p = self.params
        c = self.config

        # Time increment
        dt = c.maturity / c.n_steps

        # Initialize paths: Shape (steps + 1, paths)
        S = np.zeros((c.n_steps + 1, c.n_paths))
        v = np.zeros((c.n_steps + 1, c.n_paths))

        S[0] = p.s0
        v[0] = p.v0

        # Set seed for reproducibility
        np.random.seed(c.seed)

        # Cholesky decomposition for correlated Brownian Motions
        # Correlation matrix: [[1, rho], [rho, 1]]
        # Lower triangular L: [[1, 0], [rho, sqrt(1-rho^2)]]
        chol_v21 = p.rho
        chol_v22 = np.sqrt(1.0 - p.rho**2)

        for t in range(1, c.n_steps + 1):
            # Generate two independent sets of standard normal random variables
            z1 = np.random.normal(0, 1, c.n_paths)
            z2 = np.random.normal(0, 1, c.n_paths)

            # Correlated shocks
            dW_s = z1 * np.sqrt(dt)
            dW_v = (chol_v21 * z1 + chol_v22 * z2) * np.sqrt(dt)

            # Full Truncation: We use v_plus = max(v, 0) for the drift and diffusion
            v_prev = v[t-1]
            v_plus = np.maximum(v_prev, 0)

            # Update Variance: dv_t = kappa * (theta - v_t+) * dt + sigma * sqrt(v_t+) * dW_v
            v[t] = v_prev + p.kappa * (p.theta - v_plus) * dt + \
                   p.sigma * np.sqrt(v_plus) * dW_v

            # Update Asset Price: dlnS_t = (r - q - 0.5 * v_t+) * dt + sqrt(v_t+) * dW_s
            # We use the log-Euler scheme for S to ensure price stays positive.
            drift = (p.r - p.q - 0.5 * v_plus) * dt
            diffusion = np.sqrt(v_plus) * dW_s
            S[t] = S[t-1] * np.exp(drift + diffusion)

        return S, v

In [ ]:
# Example Usage:
# h_params = default_heston_params()
# mc_config = default_mc_config()
# simulator = HestonModelSimulator(h_params, mc_config)
# S_paths, v_paths = simulator.simulate()